Note: Due to package version differences over time, running the topic modeling script with the provided seed may yield slightly different cluster boundaries. The exact topic labels and assignments used in the published paper are provided as-is in topics_output.json for exact replication.

In [ ]:
from datasets import load_dataset
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer # 2.2.2 --> 2.7.0
from sentence_transformers.util import cos_sim

from sklearn.cluster import SpectralClustering, KMeans
from sklearn.metrics import silhouette_samples, silhouette_score, confusion_matrix, f1_score

from sklearn.decomposition import PCA

import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [31]:
import umap

print(umap.__version__)

0.5.12


In [ ]:
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

from umap import UMAP

# Load raw data

In [ ]:
ds = load_dataset("wwbrannon/twinviews-13k")

In [ ]:
SEED = 92
N_TEST = -1

if N_TEST > 0:
    try:
        ds = ds["train"].shuffle(SEED).select(range(N_TEST))
    except:
        ds = ds.shuffle(SEED).select(range(N_TEST))
else: 
    try:
        ds = ds["train"]
    except:
        pass

In [ ]:
len(ds)

In [ ]:
documents = [ # merge both stances (--> logic: same topic)
    f"{r}\n{l}"
    for r,l in zip(ds["r"],ds["l"])
]

# Topic Modeling

In [ ]:
model = SentenceTransformer("avsolatorio/GIST-Embedding-v0") # Load model

In [ ]:
# Create your representation model
representation_model = KeyBERTInspired()
vectorizer_model = CountVectorizer(stop_words="english")

umap = UMAP(
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    low_memory=False,
)

topic_model = BERTopic(
    embedding_model=model, 
    nr_topics="auto",
    umap_model=umap,
    representation_model=representation_model, 
    vectorizer_model=vectorizer_model
)

topics, probs = topic_model.fit_transform(documents)

In [ ]:
topic_model.visualize_heatmap()

## Merging topics

### Similarity based

Based on predefined similarity threshold.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from copy import deepcopy

In [ ]:
topic_model_merged = deepcopy(topic_model)

In [ ]:
SIM_THRESH = .88
print(f"Start nb topics: {len(topic_model_merged.topic_labels_)}")

In [ ]:
distance_matrix = cosine_similarity(np.array(topic_model_merged.topic_embeddings_)[1:, :])
labels = (topic_model_merged.get_topic_info().sort_values("Topic", ascending=True).Name)[1:]

In [ ]:
merge_loop_counter = 0

distance_matrix = cosine_similarity(np.array(topic_model_merged.topic_embeddings_)[1:, :])

while np.max(distance_matrix-np.identity(len(distance_matrix)))>=SIM_THRESH:
    print(f"Merging loop number {merge_loop_counter}")
    merge_loop_counter += 1

    topics_to_merge = []

    for i, row in enumerate(distance_matrix):
        truncated_row = row[:i]
        if len(truncated_row):
            if np.max(truncated_row) >= SIM_THRESH:
                topics_to_merge += [[i, np.argmax(truncated_row)]]

    topic_model_merged.merge_topics(documents, topics_to_merge)
    distance_matrix = cosine_similarity(np.array(topic_model_merged.topic_embeddings_)[1:, :])

print(f"End nb topics: {len(topic_model_merged.topic_labels_)}")

In [ ]:
topic_model_merged.visualize_heatmap()

In [ ]:
topics_per_class = topic_model.topics_per_class(documents, classes=ds["topic"])
topic_model.visualize_topics_per_class(topics_per_class, top_n_topics=20)

### Including pre-defined topics

In [ ]:
multi_topics_class = {}

for c in topics_per_class["Class"].unique():
    if len(topics_per_class[topics_per_class["Class"]==c])>1:
        multi_topics_class[c] = list(topics_per_class[topics_per_class["Class"]==c]["Name"])

In [ ]:
print(f"Number of pre-defined topics : {len(topics_per_class['Class'].unique())}\nNumber of pre-defined topics with several BERTopics: {len(multi_topics_class)}")

In [ ]:
refined_topics = pd.DataFrame()
refined_topics["predefined_topics"] = topics_per_class["Class"].unique()

max_predicted_topic = []

for pt in refined_topics["predefined_topics"]:
    curr_pt_df = topics_per_class[topics_per_class["Class"]==pt]
    if len(curr_pt_df)>1:
        max_predicted_topic += [list(curr_pt_df["Name"])[curr_pt_df["Frequency"].argmax()]]
    else:
        max_predicted_topic += [list(curr_pt_df["Name"])[0]]

refined_topics["predicted_topics"] = max_predicted_topic

In [ ]:
for pred_topic in [x for _, x in sorted(zip([int(t.split("_")[0]) for t in refined_topics["predicted_topics"].unique()], refined_topics["predicted_topics"].unique()))]:
    associated_predif = list(refined_topics[refined_topics['predicted_topics']==pred_topic]['predefined_topics'])
    print(f"===================== {pred_topic} ({len(associated_predif)}) [{np.sum([t in associated_predif for t in ds['topic']])}]\n{associated_predif}")

### Manually setting labels

Manually created `manual_labels.json` file setting manually the labels for the automatically retrieved topics.

In [ ]:
import json
import pickle

In [ ]:
with open("manual_labels.json", "r") as f:
    manual_labels = json.load(f)

In [ ]:
topic_model_merged.set_topic_labels(manual_labels)

In [ ]:
refined_topics["predicted_topics"] = refined_topics["predicted_topics"].apply(lambda k: manual_labels[k.split("_")[0]])

In [ ]:
refined_topics

In [ ]:
topic_to_bertopic = {
    row["predefined_topics"]: row["predicted_topics"]
    for _, row in refined_topics.iterrows()
}

In [ ]:
with open('data/dataprep/topic_to_bertopic.json', 'w') as f:
   json.dump(topic_to_bertopic, f, indent=4)